# 02 — Probas Niveau 1 (OOF + val/test)
Génère les probabilités de niveau 1 pour le stacking :
- **OOF (Out-Of-Fold)** sur le train → pas de leakage pour entraîner le méta-classifieur
- **Val/test** avec modèles entraînés sur tout le train

Sauvegarde dans `data/processed_clean/` et modèles dans `models/clean/`.

In [2]:
import os, sys, pickle
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms as T
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import normalize
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [3]:
BASE         = r'C:/Users/veyss/Nouveau dossier/nov25_alt1_mle_rakuten'
SPLIT_DIR    = os.path.join(BASE, 'data', 'split_no_crop')
OLD_PROC     = os.path.join(BASE, 'data', 'processed')
CLEAN_PROC   = os.path.join(BASE, 'data', 'processed_clean')
CLEAN_MODELS = os.path.join(BASE, 'models', 'clean')
OLD_MODELS   = os.path.join(BASE, 'models')

N_FOLDS   = 5
N_CLASSES = 27

P = {
    # ── Texte ──────────────────────────────────────────────────────────
    'text_oof'      : os.path.join(CLEAN_PROC, 'P_text_oof_train.npy'),
    'text_val'      : os.path.join(CLEAN_PROC, 'P_text_val.npy'),
    'text_test'     : os.path.join(CLEAN_PROC, 'P_text_test.npy'),
    # ── DINOv2 lin no-crop ────────────────────────────────────────────
    'lin_oof'       : os.path.join(CLEAN_PROC, 'P_dino_lin_oof_train.npy'),
    'lin_val'       : os.path.join(CLEAN_PROC, 'P_dino_lin_val.npy'),
    'lin_test'      : os.path.join(CLEAN_PROC, 'P_dino_lin_test.npy'),
    # ── DINOv2 lin crop ───────────────────────────────────────────────
    'lin_crop_oof'  : os.path.join(CLEAN_PROC, 'P_dino_lin_crop_oof_train.npy'),
    'lin_crop_val'  : os.path.join(CLEAN_PROC, 'P_dino_lin_crop_val.npy'),
    'lin_crop_test' : os.path.join(CLEAN_PROC, 'P_dino_lin_crop_test.npy'),
    # ── DINOv2 FT amélioré ────────────────────────────────────────────
    # ft_oof : OOF propre si TRAIN_FT_OOF=True (~2h30), sinon absent → fallback in-sample
    'ft_oof'        : os.path.join(CLEAN_PROC, 'P_dino_ft_oof_train.npy'),
    'ft_train'      : os.path.join(CLEAN_PROC, 'P_dino_ft_train.npy'),
    'ft_val'        : os.path.join(CLEAN_PROC, 'P_dino_ft_val.npy'),
    'ft_test'       : os.path.join(CLEAN_PROC, 'P_dino_ft_test.npy'),
}
print('Paths OK')

Paths OK


In [4]:
y_train = np.load(os.path.join(CLEAN_PROC, 'y_train.npy'))
y_val   = np.load(os.path.join(CLEAN_PROC, 'y_val.npy'))
y_test  = np.load(os.path.join(CLEAN_PROC, 'y_test.npy'))

train_split = pd.read_csv(os.path.join(CLEAN_PROC, 'train.csv'))
val_split   = pd.read_csv(os.path.join(CLEAN_PROC, 'val.csv'))
test_split  = pd.read_csv(os.path.join(CLEAN_PROC, 'test.csv'))

# Features texte combines
X_tfidf  = {s: sp.load_npz(os.path.join(CLEAN_PROC, f'X_tfidf_{s}.npz'))   for s in ['train','val','test']}
X_ctfidf = {s: np.load(os.path.join(CLEAN_PROC, f'X_ctfidf_{s}.npy'))      for s in ['train','val','test']}
X_e5     = {s: np.load(os.path.join(CLEAN_PROC, f'X_e5_{s}.npy'))          for s in ['train','val','test']}

X_txt = {
    s: sp.hstack([X_tfidf[s],
                  sp.csr_matrix(X_ctfidf[s]),
                  sp.csr_matrix(X_e5[s].astype('float32'))])
    for s in ['train', 'val', 'test']
}

# Features image DINOv2 (no-crop + crop)
X_dino      = {s: normalize(np.load(os.path.join(CLEAN_PROC, f'X_dino_{s}.npy')))      for s in ['train','val','test']}
X_dino_crop = {s: normalize(np.load(os.path.join(CLEAN_PROC, f'X_dino_crop_{s}.npy'))) for s in ['train','val','test']}

print(f'Texte complet    : {X_txt["train"].shape}')
print(f'DINOv2 no-crop   : {X_dino["train"].shape}')
print(f'DINOv2 crop      : {X_dino_crop["train"].shape}')

Texte complet    : (59185, 51051)
DINOv2 no-crop   : (59185, 768)
DINOv2 crop      : (59185, 768)


## 1. Probas texte — OOF train + full val/test

In [5]:
import time

# ── OOF texte (5-fold LogReg) ─────────────────────────────────────────
if os.path.exists(P['text_oof']):
    P_text_oof = np.load(P['text_oof'])
    print(f'OOF texte : charge | F1={f1_score(y_train, P_text_oof.argmax(1), average="macro"):.4f}')
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    P_text_oof = np.zeros((len(y_train), N_CLASSES), dtype='float32')
    t0_total = time.time()
    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_train)), y_train)):
        t0 = time.time()
        print(f'  Fold {fold+1}/{N_FOLDS} : entrainement LogReg ({len(tr_idx):,} exemples)...')
        clf_f = LogisticRegression(C=1.0, max_iter=300, class_weight='balanced',
                                    solver='saga', n_jobs=-1, random_state=42)
        clf_f.fit(X_txt['train'][tr_idx], y_train[tr_idx])
        P_text_oof[va_idx] = clf_f.predict_proba(X_txt['train'][va_idx]).astype('float32')
        f1 = f1_score(y_train[va_idx], P_text_oof[va_idx].argmax(1), average='macro')
        elapsed = time.time() - t0
        print(f'  Fold {fold+1}/{N_FOLDS} : F1={f1:.4f}  ({elapsed:.0f}s)')
    np.save(P['text_oof'], P_text_oof)
    total = time.time() - t0_total
    print(f'\nOOF texte : sauvegarde | F1_macro={f1_score(y_train, P_text_oof.argmax(1), average="macro"):.4f}  (total {total:.0f}s)')

# ── Full model val/test ───────────────────────────────────────────────
if os.path.exists(P['text_val']) and os.path.exists(P['text_test']):
    P_text_val  = np.load(P['text_val'])
    P_text_test = np.load(P['text_test'])
    print(f'Texte full : charge | val F1={f1_score(y_val, P_text_val.argmax(1), average="macro"):.4f}')
else:
    old_val  = os.path.join(OLD_PROC, 'probas_exp_mlp_ctfidf_st_dino_no_crop_text_val.npy')
    old_test = os.path.join(OLD_PROC, 'probas_exp_mlp_ctfidf_st_dino_no_crop_text_test.npy')
    if os.path.exists(old_val):
        P_text_val  = np.load(old_val)
        P_text_test = np.load(old_test)
        print('Texte full : charge depuis ancien cache')
    else:
        old_clf = os.path.join(OLD_MODELS, 'clf1_text_exp_mlp_ctfidf_st_dino_no_crop.pkl')
        if os.path.exists(old_clf):
            with open(old_clf, 'rb') as f: clf_full = pickle.load(f)
        else:
            print('Entrainement clf_text full (~5 min)...')
            t0 = time.time()
            clf_full = LogisticRegression(C=1.0, max_iter=300, class_weight='balanced',
                                          solver='saga', n_jobs=-1, random_state=42)
            clf_full.fit(X_txt['train'], y_train)
            print(f'clf_text full : entraine en {time.time()-t0:.0f}s')
            with open(os.path.join(CLEAN_MODELS, 'clf_text_full.pkl'), 'wb') as f: pickle.dump(clf_full, f)
        P_text_val  = clf_full.predict_proba(X_txt['val']).astype('float32')
        P_text_test = clf_full.predict_proba(X_txt['test']).astype('float32')
    np.save(P['text_val'],  P_text_val)
    np.save(P['text_test'], P_text_test)
    print(f'Texte full : val F1={f1_score(y_val, P_text_val.argmax(1), average="macro"):.4f}')

OOF texte : charge | F1=0.8372
Texte full : charge | val F1=0.8340


## 2. Probas DINOv2 linéaire — OOF train + full val/test

In [6]:
class LinearHead(nn.Module):
    def __init__(self, in_dim, n=27):
        super().__init__()
        self.fc = nn.Linear(in_dim, n)
    def forward(self, x): return self.fc(x)

def train_head(Xtr, ytr, Xva, yva, ckpt, n_epochs=40, lr=1e-3, bs=256, name=''):
    model = LinearHead(Xtr.shape[1]).to(device)
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=device))
        with torch.no_grad():
            p = model(torch.tensor(Xva, dtype=torch.float32).to(device)).argmax(1).cpu().numpy()
        f1 = f1_score(yva, p, average='macro')
        print(f'  [{name or os.path.basename(ckpt)}] ckpt charge | val F1={f1:.4f}')
        return model

    loader = DataLoader(TensorDataset(torch.tensor(Xtr, dtype=torch.float32),
                                      torch.tensor(ytr, dtype=torch.long)),
                        batch_size=bs, shuffle=True)
    Xva_t = torch.tensor(Xva, dtype=torch.float32).to(device)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    crit  = nn.CrossEntropyLoss()
    best_f1, no_imp = 0.0, 0
    tag = f'[{name}] ' if name else ''

    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss, n_batches = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            total_loss += loss.item(); n_batches += 1
        sched.step()

        model.eval()
        with torch.no_grad():
            p = model(Xva_t).argmax(1).cpu().numpy()
        f1 = f1_score(yva, p, average='macro')

        improved = f1 > best_f1
        if improved:
            best_f1, no_imp = f1, 0
            torch.save(model.state_dict(), ckpt)
        else:
            no_imp += 1

        if epoch % 10 == 0 or epoch == 1 or improved:
            mark = ' ✓' if improved else ''
            print(f'  {tag}epoch {epoch:3d}/{n_epochs} | loss={total_loss/n_batches:.4f} | val F1={f1:.4f} | best={best_f1:.4f}{mark}')

        if no_imp >= 5:
            print(f'  {tag}early stop epoch {epoch} | best F1={best_f1:.4f}')
            break

    model.load_state_dict(torch.load(ckpt, map_location=device))
    return model

def get_probas(model, X, bs=512):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb = torch.tensor(X[i:i+bs], dtype=torch.float32).to(device)
            out.append(torch.softmax(model(xb), dim=1).cpu().numpy())
    return np.vstack(out)

print('LinearHead + helpers definis')

LinearHead + helpers definis


In [7]:
# ── OOF lineaire no-crop (5-fold) ────────────────────────────────────
if os.path.exists(P['lin_oof']):
    P_lin_oof = np.load(P['lin_oof'])
    print(f'OOF DINOv2-lin : charge | F1={f1_score(y_train, P_lin_oof.argmax(1), average="macro"):.4f}')
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    P_lin_oof = np.zeros((len(y_train), N_CLASSES), dtype='float32')
    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_train)), y_train)):
        print(f'\n--- Fold {fold+1}/{N_FOLDS} (nocrop) ---')
        ckpt_f = os.path.join(CLEAN_MODELS, f'dino_linear_fold{fold+1}.pth')
        m = train_head(X_dino['train'][tr_idx], y_train[tr_idx],
                       X_dino['train'][va_idx],  y_train[va_idx], ckpt_f,
                       name=f'nocrop-fold{fold+1}')
        P_lin_oof[va_idx] = get_probas(m, X_dino['train'][va_idx])
        print(f'  => fold {fold+1} OOF F1={f1_score(y_train[va_idx], P_lin_oof[va_idx].argmax(1), average="macro"):.4f}')
    np.save(P['lin_oof'], P_lin_oof)
    print(f'\nOOF DINOv2-lin : sauvegarde | F1={f1_score(y_train, P_lin_oof.argmax(1), average="macro"):.4f}')

# ── Full val/test ─────────────────────────────────────────────────────
if os.path.exists(P['lin_val']) and os.path.exists(P['lin_test']):
    P_lin_val  = np.load(P['lin_val'])
    P_lin_test = np.load(P['lin_test'])
    print(f'DINOv2-lin full : charge | val F1={f1_score(y_val, P_lin_val.argmax(1), average="macro"):.4f}')
else:
    print('\n--- Full no-crop ---')
    ckpt_full = os.path.join(CLEAN_MODELS, 'dino_linear_full.pth')
    m_full    = train_head(X_dino['train'], y_train, X_dino['val'], y_val, ckpt_full,
                           name='nocrop-full')
    P_lin_val  = get_probas(m_full, X_dino['val'])
    P_lin_test = get_probas(m_full, X_dino['test'])
    np.save(P['lin_val'],  P_lin_val)
    np.save(P['lin_test'], P_lin_test)
    print(f'DINOv2-lin full : val F1={f1_score(y_val, P_lin_val.argmax(1), average="macro"):.4f}')

OOF DINOv2-lin : charge | F1=0.5973
DINOv2-lin full : charge | val F1=0.6117


## 2b. Probas DINOv2 linéaire crop — OOF train + full val/test

In [8]:
# ── OOF lineaire crop (5-fold) ────────────────────────────────────────
if os.path.exists(P['lin_crop_oof']):
    P_lin_crop_oof = np.load(P['lin_crop_oof'])
    print(f'OOF DINOv2-lin-crop : charge | F1={f1_score(y_train, P_lin_crop_oof.argmax(1), average="macro"):.4f}')
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    P_lin_crop_oof = np.zeros((len(y_train), N_CLASSES), dtype='float32')
    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_train)), y_train)):
        print(f'\n--- Fold {fold+1}/{N_FOLDS} (crop) ---')
        ckpt_f = os.path.join(CLEAN_MODELS, f'dino_linear_crop_fold{fold+1}.pth')
        m = train_head(X_dino_crop['train'][tr_idx], y_train[tr_idx],
                       X_dino_crop['train'][va_idx],  y_train[va_idx], ckpt_f,
                       name=f'crop-fold{fold+1}')
        P_lin_crop_oof[va_idx] = get_probas(m, X_dino_crop['train'][va_idx])
        print(f'  => fold {fold+1} OOF F1={f1_score(y_train[va_idx], P_lin_crop_oof[va_idx].argmax(1), average="macro"):.4f}')
    np.save(P['lin_crop_oof'], P_lin_crop_oof)
    print(f'\nOOF DINOv2-lin-crop : sauvegarde | F1={f1_score(y_train, P_lin_crop_oof.argmax(1), average="macro"):.4f}')

# ── Full val/test ─────────────────────────────────────────────────────
if os.path.exists(P['lin_crop_val']) and os.path.exists(P['lin_crop_test']):
    P_lin_crop_val  = np.load(P['lin_crop_val'])
    P_lin_crop_test = np.load(P['lin_crop_test'])
    print(f'DINOv2-lin-crop full : charge | val F1={f1_score(y_val, P_lin_crop_val.argmax(1), average="macro"):.4f}')
else:
    print('\n--- Full crop ---')
    ckpt_full = os.path.join(CLEAN_MODELS, 'dino_linear_crop_full.pth')
    m_full    = train_head(X_dino_crop['train'], y_train, X_dino_crop['val'], y_val, ckpt_full,
                           name='crop-full')
    P_lin_crop_val  = get_probas(m_full, X_dino_crop['val'])
    P_lin_crop_test = get_probas(m_full, X_dino_crop['test'])
    np.save(P['lin_crop_val'],  P_lin_crop_val)
    np.save(P['lin_crop_test'], P_lin_crop_test)
    print(f'DINOv2-lin-crop full : val F1={f1_score(y_val, P_lin_crop_val.argmax(1), average="macro"):.4f}')

OOF DINOv2-lin-crop : charge | F1=0.5972
DINOv2-lin-crop full : charge | val F1=0.6121


## 3. Probas DINOv2 fine-tuné — entraînement amélioré

Améliorations par rapport à la version naïve (val F1=0.7184) :

| Problème | Solution |
|---|---|
| LR uniforme → backbone détruit | **LR différentiel** : head=1e-3, blk11=5e-5, blk10=2e-5 |
| Pas de warmup | **Cosine warmup** : montée linéaire 5 epochs, puis cosine decay |
| Pas d'augmentation | **RandomResizedCrop + ColorJitter + RandomGrayscale** en train |
| Tout dégelé dès epoch 1 | **Dégel progressif** : head seul → +blk11 ep.6 → +blk10 ep.16 |
| In-sample probas (overconfident) | **OOF optionnel** (`TRAIN_FT_OOF=True`) : 5-fold ~2h30 |
| Instabilité gradient | **Gradient clipping** (max_norm=1.0) + **MixUp** (α=0.2) |

In [ ]:
from transformers import AutoModel
import math

# ── Hyperparamètres FT ────────────────────────────────────────────────
TRAIN_FT_OOF   = False   # True → 5-fold OOF ; False → single run
MIXUP_ALPHA    = 0.2
N_EPOCHS_FT    = 15
WARMUP_EPOCHS  = 3
UNFREEZE_SCHEDULE = {5: [11], 13: [10]}   # epoch → blocs DINOv2 à dégeler

LR_HEAD   = 1e-3
LR_BLK11  = 5e-5
LR_BLK10  = 2e-5

# ── Augmentations ─────────────────────────────────────────────────────
_mean, _std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

tf_train = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    T.Normalize(_mean, _std),
])
tf_val = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(_mean, _std),
])

# ── Dataset images (avec labels optionnels pour shuffle + MixUp) ──────
SPLIT_DIR_FT = os.path.join(BASE, 'data', 'split_no_crop')

class ImgDataset(Dataset):
    def __init__(self, df, split_name, transform, labels=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = os.path.join(SPLIT_DIR_FT, split_name)
        self.transform = transform
        self.labels = labels
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        try:
            img = Image.open(os.path.join(self.img_dir, self.df.iloc[i]['image_name'])).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224))
        x = self.transform(img)
        if self.labels is not None:
            return x, int(self.labels[i])
        return x

# ── Modèle DINOv2 + tête MLP GELU ────────────────────────────────────
class DINOv2Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained('facebook/dinov2-base')
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        self.head = nn.Sequential(
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, N_CLASSES),
        )
    def forward(self, pixel_values):
        cls = self.backbone(pixel_values=pixel_values).last_hidden_state[:, 0, :]
        return self.head(cls)
    def unfreeze_block(self, block_idx):
        for p in self.backbone.encoder.layer[block_idx].parameters():
            p.requires_grad_(True)
        if block_idx == 11:
            for p in self.backbone.layernorm.parameters():
                p.requires_grad_(True)

# ── MixUp ─────────────────────────────────────────────────────────────
def mixup_batch(imgs, labels, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(len(imgs), device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    soft_a = torch.zeros(len(labels), N_CLASSES, device=imgs.device).scatter_(1, labels.unsqueeze(1), 1.0)
    soft_b = soft_a[idx]
    return mixed, lam * soft_a + (1 - lam) * soft_b

# ── Fonction d'entraînement FT ────────────────────────────────────────
def cosine_warmup_scale(epoch, warmup, total):
    if epoch <= warmup:
        return epoch / warmup
    prog = (epoch - warmup) / max(total - warmup, 1)
    return 0.5 * (1 + math.cos(math.pi * prog))

def train_dino_ft(train_df, y_tr, val_df, y_va, ckpt_path, tag='', val_split_name='val'):
    """
    DINOv2 FT : LR différentiel, dégel progressif, warmup cosine, MixUp, grad clip.
    val_split_name='val'   pour le single run
    val_split_name='train' pour les folds OOF
    """
    model = DINOv2Classifier().to(device)
    prefix = f'[{tag}] ' if tag else ''

    opt = optim.AdamW([{'params': list(model.head.parameters()),
                        'lr': LR_HEAD, 'initial_lr': LR_HEAD}],
                      weight_decay=1e-4)

    best_f1, no_imp = 0.0, 0

    train_ds  = ImgDataset(train_df, 'train',        tf_train, y_tr)
    val_ds    = ImgDataset(val_df,   val_split_name, tf_val)
    train_ldr = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0, pin_memory=True)
    val_ldr   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

    for epoch in range(1, N_EPOCHS_FT + 1):

        # ── Dégel progressif ─────────────────────────────────────────
        if epoch in UNFREEZE_SCHEDULE:
            for blk_idx in UNFREEZE_SCHEDULE[epoch]:
                model.unfreeze_block(blk_idx)
                lr_blk = LR_BLK11 if blk_idx == 11 else LR_BLK10
                new_params = [p for p in model.backbone.encoder.layer[blk_idx].parameters() if p.requires_grad]
                if blk_idx == 11:
                    new_params += [p for p in model.backbone.layernorm.parameters() if p.requires_grad]
                opt.add_param_group({'params': new_params, 'lr': lr_blk, 'initial_lr': lr_blk})
                print(f'{prefix}epoch {epoch} : dégel bloc {blk_idx} (lr={lr_blk:.0e})')

        # ── LR cosine warmup ─────────────────────────────────────────
        scale = cosine_warmup_scale(epoch, WARMUP_EPOCHS, N_EPOCHS_FT)
        for pg in opt.param_groups:
            pg['lr'] = pg['initial_lr'] * scale

        # ── Entraînement ─────────────────────────────────────────────
        model.train()
        total_loss, n_batches = 0.0, 0
        for imgs, labels in train_ldr:
            imgs   = imgs.to(device)
            labels = labels.to(device)
            imgs, soft = mixup_batch(imgs, labels, MIXUP_ALPHA)
            opt.zero_grad()
            logits = model(imgs)
            loss = -(soft * torch.log_softmax(logits, dim=1)).sum(1).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            total_loss += loss.item(); n_batches += 1

        # ── Évaluation ───────────────────────────────────────────────
        model.eval()
        preds = []
        with torch.no_grad():
            for imgs in val_ldr:
                preds.append(model(imgs.to(device)).argmax(1).cpu())
        f1 = f1_score(y_va, torch.cat(preds).numpy(), average='macro')

        improved = f1 > best_f1
        if improved:
            best_f1, no_imp = f1, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_imp += 1

        mark = ' ✓' if improved else ''
        print(f'{prefix}epoch {epoch:3d}/{N_EPOCHS_FT} | loss={total_loss/n_batches:.4f} | val F1={f1:.4f} | best={best_f1:.4f}{mark}')

        if no_imp >= 5:
            print(f'{prefix}early stop epoch {epoch} | best F1={best_f1:.4f}')
            break

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return model

# ── Inférence ─────────────────────────────────────────────────────────
def get_ft_probas(model, df, split_name, bs=64):
    model.eval()
    loader = DataLoader(ImgDataset(df, split_name, tf_val),
                        batch_size=bs, shuffle=False, num_workers=0)
    out = []
    with torch.no_grad():
        for imgs in loader:
            out.append(torch.softmax(model(imgs.to(device)), dim=1).cpu().numpy())
    return np.vstack(out)

print('DINOv2Classifier + train_dino_ft + get_ft_probas définis')
print(f'TRAIN_FT_OOF={TRAIN_FT_OOF} | N_EPOCHS_FT={N_EPOCHS_FT} | WARMUP={WARMUP_EPOCHS} | MixUp α={MIXUP_ALPHA}')
print(f'UNFREEZE_SCHEDULE={UNFREEZE_SCHEDULE} | early_stop patience=5')

DINOv2Classifier + train_dino_ft + get_ft_probas définis
TRAIN_FT_OOF=False | N_EPOCHS_FT=15 | WARMUP=3 | MixUp α=0.2
UNFREEZE_SCHEDULE={5: [11], 13: [10]} | early_stop patience=5


In [13]:
all_ft_cached = (os.path.exists(P['ft_val']) and os.path.exists(P['ft_test'])
                 and os.path.exists(P['ft_train']))

# ── Mode OOF (TRAIN_FT_OOF=True) ────────────────────────────────────
if TRAIN_FT_OOF and not os.path.exists(P['ft_oof']):
    print('=== DINOv2-FT OOF 5-fold (~2h30) ===')
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    P_ft_oof = np.zeros((len(y_train), N_CLASSES), dtype='float32')
    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_train)), y_train)):
        print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
        ckpt_f  = os.path.join(CLEAN_MODELS, f'dino_ft_fold{fold+1}.pth')
        df_tr_f = train_split.iloc[tr_idx].reset_index(drop=True)
        df_va_f = train_split.iloc[va_idx].reset_index(drop=True)
        # val_split_name='train' : les images du fold val sont dans split_no_crop/train/
        m = train_dino_ft(df_tr_f, y_train[tr_idx], df_va_f, y_train[va_idx],
                          ckpt_f, tag=f'fold{fold+1}', val_split_name='train')
        P_ft_oof[va_idx] = get_ft_probas(m, df_va_f, 'train')
        f1_f = f1_score(y_train[va_idx], P_ft_oof[va_idx].argmax(1), average='macro')
        print(f'  => fold {fold+1} OOF F1={f1_f:.4f}')
        del m; torch.cuda.empty_cache()
    np.save(P['ft_oof'], P_ft_oof)
    print(f'\nFT OOF : sauvegarde | F1={f1_score(y_train, P_ft_oof.argmax(1), average="macro"):.4f}')
elif os.path.exists(P['ft_oof']):
    P_ft_oof = np.load(P['ft_oof'])
    print(f'FT OOF : charge | F1={f1_score(y_train, P_ft_oof.argmax(1), average="macro"):.4f}')
else:
    P_ft_oof = None
    print('FT OOF : non disponible (TRAIN_FT_OOF=False) → fallback in-sample pour méta')

# ── Single run : train/val/test ──────────────────────────────────────
if all_ft_cached:
    P_ft_train = np.load(P['ft_train'])
    P_ft_val   = np.load(P['ft_val'])
    P_ft_test  = np.load(P['ft_test'])
    print(f'DINOv2-FT single : charge | val F1={f1_score(y_val, P_ft_val.argmax(1), average="macro"):.4f}')
else:
    print('\n=== DINOv2-FT single run (train complet) ===')
    ckpt_full = os.path.join(CLEAN_MODELS, 'dino_ft_full.pth')
    # val_split_name par défaut = 'val' (val_split images dans split_no_crop/val/)
    model_ft  = train_dino_ft(train_split, y_train, val_split, y_val,
                               ckpt_full, tag='full')
    P_ft_train = get_ft_probas(model_ft, train_split, 'train')
    P_ft_val   = get_ft_probas(model_ft, val_split,   'val')
    P_ft_test  = get_ft_probas(model_ft, test_split,  'test')
    np.save(P['ft_train'], P_ft_train)
    np.save(P['ft_val'],   P_ft_val)
    np.save(P['ft_test'],  P_ft_test)
    print(f'DINOv2-FT single : sauvegarde | val F1={f1_score(y_val, P_ft_val.argmax(1), average="macro"):.4f}')
    del model_ft; torch.cuda.empty_cache()

# P_ft_oof = OOF propres si TRAIN_FT_OOF=True, sinon in-sample fallback
if P_ft_oof is None:
    P_ft_oof = P_ft_train
    print('⚠ Méta-train FT utilisera probas in-sample (distribution mismatch attendu)')

FT OOF : non disponible (TRAIN_FT_OOF=False) → fallback in-sample pour méta

=== DINOv2-FT single run (train complet) ===


KeyboardInterrupt: 

In [ ]:
with open(os.path.join(CLEAN_PROC, 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)

print('\n' + '='*65)
print('PROBAS DISPONIBLES — data/processed_clean/')
print('='*65)
rows = [
    ('P_text_oof_train',       y_train, np.load(P['text_oof'])),
    ('P_text_val',             y_val,   np.load(P['text_val'])),
    ('P_dino_lin_oof',         y_train, np.load(P['lin_oof'])),
    ('P_dino_lin_val',         y_val,   np.load(P['lin_val'])),
    ('P_dino_lin_crop_oof',    y_train, np.load(P['lin_crop_oof'])),
    ('P_dino_lin_crop_val',    y_val,   np.load(P['lin_crop_val'])),
    ('P_dino_ft_train',        y_train, P_ft_train),
    ('P_dino_ft_val',          y_val,   P_ft_val),
]
if os.path.exists(P['ft_oof']):
    rows.insert(-2, ('P_dino_ft_oof_train (OOF)', y_train, np.load(P['ft_oof'])))

for name, y, proba in rows:
    tag = ' [OOF]' if 'oof' in name.lower() else ''
    print(f'  {name:<32} F1={f1_score(y, proba.argmax(1), average="macro"):.4f}{tag}')

print('='*65)
print('Pret pour 03_experiments.ipynb')


PROBAS DISPONIBLES — data/processed_clean/


NameError: name 'P_ft_train' is not defined